# 📖 Notebook 3: Configuration ManagementWhen you have 50 servers running your app, how do you change a configuration value (like a feature flag or rate limit) across all of them? You can't SSH into each server and edit a file — that's slow and error-prone.In this notebook, we'll see three approaches:| Approach | Method | Problem ||----------|--------|---------|| 🔴 Bad | Local config files | Manual updates, servers get out of sync || 🟡 Better | Shared database config | Requires polling, slow to propagate || 🟢 Best | ZooKeeper watches | Instant push notifications to all servers |## Learning ObjectivesBy the end of this notebook, you'll understand:- Why config management is hard with multiple servers- The difference between polling and push-based config updates- How ZooKeeper watches provide instant notifications- How to build a real-time config system with ZooKeeper

## 🛠️ SetupStart the ZooKeeper ensemble first:```bashcd deep-dives/zookeeperdocker-compose up -d```### Kernel SelectionSelect the `.venv` kernel in VS Code's kernel picker (top-right of notebook).If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import threadingimport timeimport jsonimport copy

---## 🔴 Bad: Local Config Files### The ProblemThe simplest approach: each server has its own config file. To change a setting, you update the file on each server.```Server A: config.json → {"rate_limit": 100}Server B: config.json → {"rate_limit": 100}Server C: config.json → {"rate_limit": 100}```Want to change the rate limit to 200? You need to update 3 files. But what if Server C is unreachable when you deploy?```Server A: config.json → {"rate_limit": 200}  ✅ updatedServer B: config.json → {"rate_limit": 200}  ✅ updatedServer C: config.json → {"rate_limit": 100}  ❌ still old!```Now your servers are out of sync. Users hitting Server C get a different experience.

In [ ]:
class LocalConfigServer:    """    BAD: Each server has its own local config.    Config changes require updating each server individually.    """    def __init__(self, server_id, config):        self.server_id = server_id        # Each server gets its own COPY of the config        self.config = copy.deepcopy(config)    def get_config(self, key):        return self.config.get(key)    def update_config(self, key, value):        """Update this server's local config (like editing a file)."""        self.config[key] = value# Initial config shared by all serversinitial_config = {    "rate_limit": 100,    "feature_dark_mode": False,    "max_upload_mb": 10}# Create 3 servers with the same initial configservers = [LocalConfigServer(i, initial_config) for i in range(3)]print("Initial state — all servers have the same config:")for s in servers:    print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')}")# Admin wants to change rate_limit to 200print("\n🔧 Admin updates rate_limit to 200...")print("   Updating Server-0... ✅")servers[0].update_config("rate_limit", 200)print("   Updating Server-1... ✅")servers[1].update_config("rate_limit", 200)print("   Updating Server-2... ❌ Network timeout! Server unreachable.")# servers[2] doesn't get updated — simulates a failed deploymentprint("\nAfter update — servers are out of sync:")for s in servers:    status = "✅" if s.get_config('rate_limit') == 200 else "❌ STALE"    print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')} {status}")print("\n❌ CONFIG DRIFT! Server-2 still has the old rate limit.")print("   Users hitting Server-2 get 100 requests/sec while others get 200.")

### Why Local Config Files Are Bad| Problem | Impact ||---------|--------|| Manual updates | Slow, error-prone, doesn't scale || Config drift | Servers have different configs || No rollback | Hard to undo a bad change || Requires restart | Most apps need a restart to pick up new config |

---## 🟡 Better: Shared Database Config### The IdeaStore config in a shared database. All servers read from the same source. To update, change it once in the database.But how do servers know when the config changes? They have to **poll** — check the database periodically.### Why It's Better- Single source of truth (no config drift)- Update once, all servers eventually get it### Why It's Still Not Great- **Polling delay**: Servers check every N seconds, so there's a gap- **Wasted resources**: Most polls find no changes- **Database load**: 50 servers polling every 5 seconds = 10 queries/sec for nothing

In [ ]:
class ConfigDatabase:    """    Simulates a shared database that stores configuration.    In production, this would be PostgreSQL, MySQL, etc.    """    def __init__(self):        self._config = {}        self._lock = threading.Lock()        self.poll_count = 0  # track how many times servers poll    def set(self, key, value):        with self._lock:            self._config[key] = value    def get_all(self):        with self._lock:            self.poll_count += 1            return copy.deepcopy(self._config)class PollingConfigServer:    """    BETTER: Reads config from a shared database.    Polls the database every `poll_interval` seconds.    """    def __init__(self, server_id, db, poll_interval=2):        self.server_id = server_id        self.db = db        self.poll_interval = poll_interval        self.local_config = {}        self.running = True        self.last_update = None    def start_polling(self):        """Background thread that polls the database for config changes."""        while self.running:            new_config = self.db.get_all()            if new_config != self.local_config:                self.local_config = new_config                self.last_update = time.time()            time.sleep(self.poll_interval)    def get_config(self, key):        return self.local_config.get(key)# Set up shared database with initial configdb = ConfigDatabase()db.set("rate_limit", 100)db.set("feature_dark_mode", False)# Create 3 servers that poll the databasepoll_interval = 2  # secondspolling_servers = [PollingConfigServer(i, db, poll_interval) for i in range(3)]# Start polling in background threadspoll_threads = []for s in polling_servers:    t = threading.Thread(target=s.start_polling, daemon=True)    t.start()    poll_threads.append(t)# Wait for initial polltime.sleep(1)print("Database Polling Config")print("=" * 60)print(f"Poll interval: {poll_interval}s")print(f"\nInitial config (all servers):")for s in polling_servers:    print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')}")# Update config in databaseprint(f"\n🔧 Admin updates rate_limit to 500 in the database...")update_time = time.time()db.set("rate_limit", 500)# Check immediately — servers haven't polled yet!print(f"\nImmediately after update:")for s in polling_servers:    val = s.get_config('rate_limit')    status = '❌ stale' if val != 500 else '✅'    print(f"  Server-{s.server_id}: rate_limit={val} {status}")# Wait for polling to catch upprint(f"\nWaiting {poll_interval + 1}s for polling to catch up...")time.sleep(poll_interval + 1)print(f"\nAfter waiting:")for s in polling_servers:    delay = round(s.last_update - update_time, 2) if s.last_update and s.last_update > update_time else "?"    print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')} ✅ (detected after ~{delay}s)")print(f"\n📊 Total database polls so far: {db.poll_count}")print(f"   Most of these polls found NO changes — wasted work!")# Stop pollingfor s in polling_servers:    s.running = False

### Why Polling Config Is Not Ideal| Limitation | Why It Matters ||------------|----------------|| Propagation delay | Servers get updates after `poll_interval` seconds || Wasted queries | 99% of polls find no changes || Harder to scale | More servers = more DB load from polling || Tradeoff | Short interval = more load; Long interval = more delay |

---## 🟢 Best: ZooKeeper Watches### The SolutionZooKeeper **watches** let servers say: "Notify me when this data changes." Instead of polling, servers are **pushed** updates instantly.```Polling (database):                Watch (ZooKeeper):                                   Server → DB: "Any changes?"        ZooKeeper → Server: "Config changed!"DB → Server: "No."                 (only when there IS a change)Server → DB: "Any changes?"        DB → Server: "No."                 Server → DB: "Any changes?"        DB → Server: "Yes! Here's the new data."  ```### Why This Is Better| Feature | Database Polling | ZooKeeper Watches ||---------|-----------------|-------------------|| Update speed | Seconds (poll interval) | Milliseconds (push) || Server load | High (constant queries) | Near-zero (event-driven) || Scaling | Gets worse with more servers | Stays efficient || Consistency | Eventual (within poll interval) | Near-instant |

In [ ]:
from kazoo.client import KazooClient# Connect to our 3-node ZooKeeper ensemblezk = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")zk.start()print("Connected to ZooKeeper!")

In [ ]:
class ZooKeeperConfigServer:    """    BEST: Uses ZooKeeper watches for instant config updates.    - No polling — the server is notified when config changes    - Near-instant propagation (milliseconds, not seconds)    - Zero wasted network calls    """    def __init__(self, server_id, zk_hosts, config_path):        self.server_id = server_id        self.config_path = config_path        self.local_config = {}        self.update_times = []  # track when we receive updates        # Each server gets its own ZooKeeper connection        self.zk = KazooClient(hosts=zk_hosts)        self.zk.start()    def start_watching(self):        """Set up a watch on the config ZNode."""        @self.zk.DataWatch(self.config_path)        def watch_config(data, stat):            """            This function is called AUTOMATICALLY by ZooKeeper whenever            the data at config_path changes. No polling needed!            """            if data:                new_config = json.loads(data.decode())                self.local_config = new_config                self.update_times.append(time.time())                print(f"  🔔 Server-{self.server_id} received config update: {new_config}")    def get_config(self, key):        return self.local_config.get(key)    def stop(self):        self.zk.stop()# Set up the config in ZooKeeperconfig_path = "/demo/config/app"zk.ensure_path("/demo/config")initial_zk_config = {    "rate_limit": 100,    "feature_dark_mode": False,    "max_upload_mb": 10}# Create the config ZNode with initial dataif zk.exists(config_path):    zk.set(config_path, json.dumps(initial_zk_config).encode())else:    zk.create(config_path, json.dumps(initial_zk_config).encode())print("ZooKeeper Watch-Based Config")print("=" * 60)# Create 3 servers that watch the configzk_hosts = "localhost:2181,localhost:2182,localhost:2183"watch_servers = [ZooKeeperConfigServer(i, zk_hosts, config_path) for i in range(3)]print("\nServers start watching config (initial values delivered):")for s in watch_servers:    s.start_watching()time.sleep(1)

In [ ]:
# Now update the config — all servers get notified instantly!print("\n🔧 Admin updates rate_limit to 500...")update_time = time.time()new_config = {    "rate_limit": 500,    "feature_dark_mode": False,    "max_upload_mb": 10}zk.set(config_path, json.dumps(new_config).encode())# Give watches a moment to firetime.sleep(1)print("\nAll servers received the update:")for s in watch_servers:    if len(s.update_times) >= 2:  # initial + update        delay_ms = round((s.update_times[-1] - update_time) * 1000)        print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')} ✅ (received in ~{delay_ms}ms)")    else:        print(f"  Server-{s.server_id}: rate_limit={s.get_config('rate_limit')}")

In [ ]:
# Let's enable a feature flag — dark mode!print("\n🔧 Admin enables dark mode feature flag...")new_config["feature_dark_mode"] = Truezk.set(config_path, json.dumps(new_config).encode())time.sleep(1)print("\nAll servers got the feature flag update:")for s in watch_servers:    dark_mode = s.get_config('feature_dark_mode')    status = '✅ enabled' if dark_mode else '❌'    print(f"  Server-{s.server_id}: dark_mode={dark_mode} {status}")print("\n🎉 All 3 servers updated simultaneously, no polling, no manual work!")

### 🔍 How Watches Work Under the Hood1. Server calls `zk.get("/config", watch=True)` — reads data AND registers a one-time watch2. When the data changes, ZooKeeper **pushes** a notification to all watchers3. The client re-registers the watch to keep receiving updates (kazoo's `DataWatch` does this automatically)```Timeline:                                   Server A ──watch──► ZooKeeper       Server B ──watch──►    │            Server C ──watch──►    │                                   │            Admin updates config ──►                                   │            ZooKeeper ──notify──► Server A                ──notify──► Server B     (all notified simultaneously!)          ──notify──► Server C      ```

In [ ]:
# Compare: how many network calls did each approach use?print("📊 Network Efficiency Comparison")print("=" * 60)print()print("Scenario: 3 servers, 2 config changes over 10 seconds")print()print("🔴 Local files:    6 manual SSH connections (2 changes × 3 servers)")print(f"🟡 Database poll:  {db.poll_count} polling queries (most found no changes)")print("🟢 ZooKeeper:      2 notifications × 3 servers = 6 messages (only when changes happen!)")print()print("With 50 servers and polling every 2 seconds:")print("🟡 Database:  ~250 queries/min (50 servers × 60s / 2s poll interval)")print("🟢 ZooKeeper: Only 50 messages per config change (one notification per server)")

---## 📊 Summary: Bad → Better → Best| | 🔴 Local Config Files | 🟡 Database Polling | 🟢 ZooKeeper Watches ||---|---|---|---|| **Consistency** | ❌ Config drift | ✅ Eventually consistent | ✅ Near-instant || **Update speed** | Minutes (manual) | Seconds (poll interval) | Milliseconds (push) || **Server load** | None | High (constant polling) | Near-zero || **Scalability** | ❌ Doesn't scale | ⚠️ More servers = more DB load | ✅ Efficient at any scale || **Complexity** | Low | Medium | Medium (needs ZK cluster) |### Key TakeawayZooKeeper **watches** flip the model from "servers ask for updates" to "servers are told about updates." This is more efficient and faster:- **Polling**: Server → Database: "Any changes?" (repeated constantly)- **Watch**: ZooKeeper → Server: "Here's the new config!" (only when it changes)### When to Use ThisUse ZooKeeper config management when:- You need config changes to propagate to all servers immediately- You have many servers and can't afford constant polling- You want a central source of truth for configurationExamples: feature flags, rate limits, database connection strings, A/B test settings

In [ ]:
# Cleanupfor s in watch_servers:    s.stop()if zk.exists("/demo"):    zk.delete("/demo", recursive=True)zk.stop()print("Cleaned up ZooKeeper nodes. Done!")